## Config

In [12]:
import os, json, re, math
from pathlib import Path
from time import time
from tqdm import tqdm

import torch
import torch.nn as nn
import tokenizers
from datasets import load_dataset, load_from_disk

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"*** Using device: {device.type} ***")

colab = False

if colab:
  from google.colab import drive
  drive.mount('/content/drive')
  ROOT_PATH = Path("/content/drive/MyDrive/ML-Projects/CausalLSTM")
else:
  ROOT_PATH = Path.cwd()

paths = {
        "DATASET_PROCESSED": ROOT_PATH / "data/wikitext2_processed",
        "DATASET_WT2": ROOT_PATH / "data/wikitext-2",
        "CONFIG": ROOT_PATH / "config",
        "TOKENIZER": ROOT_PATH / "tokenizers",
        "MODELS": ROOT_PATH / "models",
    }

for key, path in paths.items():
        path.mkdir(parents=True, exist_ok=True)

SEED = 42

*** Using device: cuda ***


In [138]:
### WRITE CONFIG FILE ###

config = {
    "vocab_size": 6735,
    "seq_len": 64,
    "batch_size": 64,
    "n_epochs": 100,
    "enable_mixed_precision": True if device.type == "cuda" else False,
    "grad_clip_norm": 0.5,
    "early_stopping_patience": 3,
    "early_stopping_epsilon": 3e-4,
    "model_params": {
        "embedding_dim": 512,
        "hidden_dim": 512,
        "num_layers": 2,
        "lstm_dropout_p": 0.6,
        "emb_dropout_p": 0.6,
        "out_dropout_p": 0.6,
    },
    "optimizer_params": {
        "lr": 2e-3,
        "weight_decay": 2e-5,
    },
    "lr_scheduler_params": {
        "factor": 0.5,
        "patience": 2,
        "threshold": 2e-3
    }
}

with open(paths["CONFIG"] / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f)
print(f"*** config saved to {paths["CONFIG"]} ***")

*** config saved to /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/config ***


## Tokenization

In [14]:
with open("data/the_modern_prometheus.txt", "r") as f:
    corpus = f.read()

In [15]:
### TRAIN & SAVE TOKENIZER ###
from tqdm import tqdm
from tokenizers import Tokenizer, Regex
from tokenizers.models import BPE, WordLevel
from tokenizers.trainers import BpeTrainer, WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKD, Lowercase, Replace, Sequence, StripAccents
from tokenizers.processors import TemplateProcessing

def tokenizer_trainer(
    training_iterator,
    vocab_size: int,
    save_path: str,
    special_tokens: list = ["<pad>", "<unk>", "<eos>"]
):
    tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    tokenizer.normalizer = Lowercase()
    # define special tokens template
    tokenizer.post_processor = TemplateProcessing(
        single="$0 <eos>",
        pair="$A <eos> $B:1 <eos>:1",
        special_tokens=[("<eos>", 2)],
    )

    trainer = WordLevelTrainer(
        vocab_size = vocab_size,
        special_tokens = special_tokens,
        show_progress = False
    )

    tokenizer.train_from_iterator(training_iterator, trainer)
    print(f"*** vocab size: {tokenizer.get_vocab_size():,} ***")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    tokenizer.save(save_path)
    print(f"*** trained tokenizer saved to: {save_path} ***")

tokenizer_trainer([corpus], config["vocab_size"], str(paths["TOKENIZER"] / "tokenizer.json"))

*** vocab size: 6,735 ***
*** trained tokenizer saved to: /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/tokenizers/tokenizer.json ***


In [16]:
### PREPARE & TOKENIZE MAIN DATASET ###
tokenizer = tokenizers.Tokenizer.from_file(str(paths["TOKENIZER"] / "tokenizer.json"))

text = corpus[:500]
tokens = tokenizer.encode(text, add_special_tokens=True).tokens
print(text)
print("="*89)
print(tokens)
print("="*89)
print(f"count of tokens: {len(tokens)}")

﻿I am by birth a Genevese, and my family is one of the most
distinguished of that republic. My ancestors had been for many years
counsellors and syndics, and my father had filled several public
situations with honour and reputation. He was respected by all who
knew him for his integrity and indefatigable attention to public
business. He passed his younger days perpetually occupied by the
affairs of his country; a variety of circumstances had prevented his
marrying early, nor was it until the dec
['\ufeff', 'i', 'am', 'by', 'birth', 'a', 'genevese', ',', 'and', 'my', 'family', 'is', 'one', 'of', 'the', 'most', 'distinguished', 'of', 'that', 'republic', '.', 'my', 'ancestors', 'had', 'been', 'for', 'many', 'years', 'counsellors', 'and', 'syndics', ',', 'and', 'my', 'father', 'had', 'filled', 'several', 'public', 'situations', 'with', 'honour', 'and', 'reputation', '.', 'he', 'was', 'respected', 'by', 'all', 'who', 'knew', 'him', 'for', 'his', 'integrity', 'and', 'indefatigable', 'attenti

In [19]:
corpus_ids = tokenizer.encode(corpus).ids

## Stream Dataset

In [11]:
%%writefile src/dataset.py
import torch
from torch.utils.data import Dataset
from itertools import chain

class StreamLMDataset(Dataset):
    """returns batches!"""
    def __init__(self, corpus_ids, batch_size=256, seq_len=128):
        full_seq = torch.tensor(corpus_ids, dtype=torch.long)

        # trim to multiple of batch_size
        stream_len = full_seq.size(0) // batch_size
        full_seq = full_seq[:stream_len * batch_size]
        full_seq = full_seq.view(batch_size, stream_len)

        self.seq_len = seq_len
        self.batch_size = batch_size
        self.full_seq = full_seq
        self.stream_len = full_seq.size(1) - 1

    def __len__(self):
        """number of batches"""
        # iterate in range len(ds) in training loop to get batches
        return self.stream_len // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        x = self.full_seq[:, start:end]
        y = self.full_seq[:, start+1:end+1]
        return x, y

Overwriting src/dataset.py


## Model

In [18]:
%%writefile src/model.py
import torch
import torch.nn as nn

class CausalLSTM(nn.Module):
    def __init__(
        self, vocab_size, embedding_dim=512, hidden_dim=1024, num_layers=2, lstm_dropout_p=0.4, emb_dropout_p=0.25, out_dropout_p=0.5
    ):
        super().__init__()
        self.hidden_dim = embedding_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.emb_dropout = nn.Dropout(emb_dropout_p)
        self.lstm = nn.LSTM(
            embedding_dim, embedding_dim, num_layers, batch_first=True, dropout = lstm_dropout_p
        )
        self.out_dropout = nn.Dropout(out_dropout_p)
        self.fc = nn.Linear(embedding_dim, vocab_size)
        self.fc.weight = self.embedding.weight
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        nn.init.uniform_(self.embedding.weight, -initrange, initrange)
        nn.init.zeros_(self.fc.bias)

    def init_hidden(self, batch_size):
        w = next(self.parameters())
        h_0 = w.new_zeros((self.num_layers, batch_size, self.hidden_dim))
        c_0 = w.new_zeros((self.num_layers, batch_size, self.hidden_dim))
        return (h_0, c_0)

    def forward(self, x, hidden):
        x = self.embedding(x) # (N, L, E)
        x = self.emb_dropout(x)
        x, hidden = self.lstm(x, hidden) # (N, L, H), ((num_layers, N, H), (num_layers, N, H))
        x = self.out_dropout(x)
        return self.fc(x).permute(0, 2, 1), hidden # (N, vocab_size, L), ((num_layers, N, H), (num_layers, N, H))

def detach_hidden(hidden):
    "detaches hidden from current graph"
    if isinstance(hidden, torch.Tensor):
        return hidden.detach()
    else:
        return tuple(detach_hidden(h) for h in hidden)

Overwriting src/model.py


## Training

In [139]:
### LOAD & PREPARE DATASET ###
from src.dataset import StreamLMDataset

train_ds = StreamLMDataset(corpus_ids, config["batch_size"], config["seq_len"])

print(f"*** batch size: {train_ds.batch_size} | sequence lenght: {train_ds.seq_len} ***")
print(f"*** count of training batches: {len(train_ds)} ***")
print(f"*** stream lenght of train set: {train_ds.stream_len}***")

*** batch size: 64 | sequence lenght: 64 ***
*** count of training batches: 19 ***
*** stream lenght of train set: 1235***


In [140]:
### LOAD & PREPARE MODEL ###
from src.model import CausalLSTM, detach_hidden

torch.manual_seed(SEED)
model = CausalLSTM(config["vocab_size"], **config["model_params"]).to(device)
print(f"*** total count of trainable parameters: {sum([p.numel() for p in model.parameters() if p.requires_grad]):,} ***")
print(model)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters(), **config["optimizer_params"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, **config["lr_scheduler_params"])

*** total count of trainable parameters: 7,657,551 ***
CausalLSTM(
  (embedding): Embedding(6735, 512)
  (emb_dropout): Dropout(p=0.6, inplace=False)
  (lstm): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.6)
  (out_dropout): Dropout(p=0.6, inplace=False)
  (fc): Linear(in_features=512, out_features=6735, bias=True)
)


In [141]:
@torch.no_grad()
def evaluate(eval_dataset, disable_progress_bar=True):
    model.eval()
    total_loss = 0.0
    hidden = model.init_hidden(config["batch_size"])
    for idx in tqdm(range(len(eval_dataset)), disable = disable_progress_bar):
        X, Y = eval_dataset[idx]
        X, Y = X.to(device), Y.to(device)
        hidden = detach_hidden(hidden)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=config["enable_mixed_precision"]):
            logits, hidden = model(X, hidden)
            total_loss += loss_fn(logits, Y).item()
    return total_loss / len(eval_dataset)

In [142]:
def train(saved_checkpoint_path = None):
    if saved_checkpoint_path is not None:
        model.load_state_dict(
            torch.load(saved_checkpoint_path, map_location=device, weights_only=True)
            )
    train_logs = {"train_loss":[] , "val_loss":[] , "val_metric":[], "lr":[]}
    model.train()
    scaler = torch.amp.GradScaler(enabled = config["enable_mixed_precision"])
    best_loss, es_counter = float('inf'), 0

    for epoch in range(config["n_epochs"]):
        start_time = time()
        model.train()
        total_loss = 0.0
        hidden = model.init_hidden(config["batch_size"])

        for idx in tqdm(range(len(train_ds)), desc=f"Epoch {epoch+1}/{config["n_epochs"]}"):
            X, Y = train_ds[idx]
            X, Y = X.to(device), Y.to(device)
            optimizer.zero_grad(set_to_none=True)
            # detach hidden from current graph
            hidden = detach_hidden(hidden)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled = config["enable_mixed_precision"]):
                logits, hidden = model(X, hidden)
                loss = loss_fn(logits, Y)
            total_loss += loss.item()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # clip gradients to avoid exloding
            nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip_norm"])
            scaler.step(optimizer)
            scaler.update()

        # logger
        train_logs["train_loss"].append(total_loss / len(train_ds))
        train_logs["lr"].append(optimizer.param_groups[0]['lr'])

        print(f"\r Epoch {epoch + 1}/{config["n_epochs"]}", end="")
        print(f", train loss: {train_logs["train_loss"][-1]:.4f}", end="")
        print(f", train perplexity: {math.exp(train_logs["train_loss"][-1]):.4f}", end="")
        print(f", lr: {train_logs["lr"][-1]}", end="")
        print(f', epoch time: {time() - start_time:.2f}s')

        torch.save(model.state_dict(), paths["MODELS"] / f"CausualLSTM_ckpnt_{epoch+1}.pt")

    return model, train_logs

In [143]:
ckpnt = str(paths["MODELS"] / f"CausualLSTM_ckpnt_{59}.pt")
model, train_logs = train()

with open(os.path.join(ROOT_PATH, "training_logs.json"), "w") as f:
    json.dump(train_logs, f)

Epoch 1/100: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 16.11it/s]


 Epoch 1/100, train loss: 7.0153, train perplexity: 1113.5788, lr: 0.002, epoch time: 1.18s


Epoch 2/100: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:00<00:00, 22.58it/s]


 Epoch 2/100, train loss: 6.2501, train perplexity: 518.0696, lr: 0.002, epoch time: 0.84s


Epoch 3/100: 100%|██████████████████████████████████████████████████████████████████████| 19/19 [00:00<00:00, 22.49it/s]


...

 Epoch 98/100, train loss: 2.0691, train perplexity: 7.9175, lr: 0.002, epoch time: 0.85s


Epoch 99/100: 100%|█████████████████████████████████████████████████████████████████████| 19/19 [00:00<00:00, 23.01it/s]


 Epoch 99/100, train loss: 2.0554, train perplexity: 7.8102, lr: 0.002, epoch time: 0.83s


Epoch 100/100: 100%|████████████████████████████████████████████████████████████████████| 19/19 [00:00<00:00, 23.26it/s]


 Epoch 100/100, train loss: 2.0441, train perplexity: 7.7220, lr: 0.002, epoch time: 0.82s


## Inference

In [144]:
import tokenizers, json, torch
from src.model import CausalLSTM

checkpoint_path = "models/CausualLSTM_ckpnt_100.pt"
config_path = "config/config.json"
tokenizer_path = "tokenizers/tokenizer.json"

with open(config_path, "r") as f:
    config = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tokenizers.Tokenizer.from_file(tokenizer_path)
model = CausalLSTM(config["vocab_size"], **config["model_params"]).to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [145]:
def generate(model, tokenizer, config, device, init_word, max_new_tokens=50, temperature=1.0, seed=42):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    hidden = model.init_hidden(1)
    unk_id, eos_id = tokenizer.token_to_id("<unk>"), tokenizer.token_to_id("<eos>")
    init_idx = tokenizer.token_to_id(init_word)
    if init_idx is not None:
        input_ = torch.tensor(init_idx, dtype=torch.long).reshape(1,-1).to(device)
    else:
        raise Exception(f"{init_word} is not in vocab")
        
    generated_words = [init_word]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            output, hidden = model(input_, hidden)
            probs = output.squeeze().div(temperature).exp().cpu()
            while True:
                token_idx = torch.multinomial(probs, 1)[0]
                if token_idx != unk_id: break
            input_.fill_(token_idx)
            generated_words.append(tokenizer.id_to_token(token_idx) if token_idx != eos_id else "\n")
    return " ".join(generated_words)

In [148]:
print(generate(model, tokenizer, config, device, init_word = "hear", max_new_tokens=62, temperature=0.7, seed=45))

hear him ; we are innocent . since the murderer of your parents been moved with my representations and _i can be ignorant of your own fair and a creator , but you will not be consulted from him ; you will pardon me to pieces that you are to suffer happy ; but it will not be consulted more than to the
